<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/866_PCFDv2_DataLoading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Loading Layer

### Product–Customer Fit Discovery Orchestrator v2 (PCFDv2)

Before an orchestrator can discover opportunities, it must solve a very practical problem:

> **How do we reliably load and verify all the data the system depends on?**

This module handles that responsibility.

It loads the full **Product–Customer Fit Discovery dataset architecture**, normalizes the data into a consistent structure, and produces metadata that ensures the system is **traceable and auditable**.

In many AI systems, data ingestion is treated as a technical afterthought.

In production environments, however, **data loading is one of the most critical parts of the system**.

If data pipelines are unreliable, every downstream insight becomes questionable.

This module avoids that risk by enforcing **structured loading, validation, and traceability**.

---

# 🧭 Role in the Orchestrator Architecture

This utility acts as the **gateway between raw datasets and the agent workflow**.

It feeds the orchestrator the structured inputs required for discovery.

The overall flow looks like this:

```
File System
    ↓
Data Loader
    ↓
Normalized Dataset Dictionary
    ↓
Agent State
    ↓
Discovery Nodes
```

By consolidating all ingestion logic into one module, the system achieves several advantages:

• consistent dataset handling
• centralized validation
• easier debugging
• reproducible runs

This design significantly improves the **operational reliability of the agent**.

---

# 📂 Path Resolution and Environment Flexibility

One of the first tasks performed by the loader is resolving the correct **data directory path**.

The `_resolve_data_root()` function allows the orchestrator to work in multiple environments:

• local development
• orchestrator registry execution
• production pipeline deployment

If a `project_root` is provided, the loader resolves paths relative to it.

Otherwise it derives the root directory dynamically from the module location.

This flexibility ensures the agent can run consistently regardless of **where the execution environment lives**.

---

# 📊 Structured Data Loading

Two helper functions perform the actual data ingestion:

### `_load_csv()`

Reads CSV files into a list of dictionaries.

Each row becomes a dictionary representing one record.

This format makes the data easy to:

• iterate over
• filter
• join with other datasets

Importantly, if a file does not exist the loader returns an **empty list instead of failing**.

This prevents the orchestrator from crashing and allows the system to detect the issue gracefully.

---

### `_load_json()`

Handles JSON datasets.

The loader supports both:

• lists of records
• single JSON objects

This flexibility allows the system to work with a variety of data structures without requiring changes to the pipeline.

---

# 📊 Loading the Data Architecture

The main function `load_all_pcfd_data()` loads every dataset used by the orchestrator.

The data is organized according to the **four-layer architecture defined in the project**.

---

## Baseline Layer

These datasets represent the operational system.

```
customers.csv
product_catalog.csv
transactions.csv
```

They define the core relationships between customers and products.

Without this layer the orchestrator would have **no behavioral signals to analyze**.

---

## Enrichment Layer

These datasets provide additional business context.

```
customer_metrics.csv
product_metrics.csv
feature_usage.csv
customer_journeys.json
market_signals.json
```

They allow the system to analyze:

• customer value
• product profitability
• feature behavior
• lifecycle patterns
• external market signals

This layer turns basic product usage data into **strategic intelligence inputs**.

---

## Trend Intelligence

Historical datasets provide time-series signals.

```
product_adoption_history.csv
segment_growth_history.csv
feature_demand_history.csv
```

These allow the orchestrator to detect:

• emerging products
• declining products
• expanding customer segments
• accelerating feature demand

Trend analysis is what transforms the system from **static analytics** into a **discovery engine**.

---

## Strategic Signals

Finally, the loader imports opportunity signals:

```
bundle_opportunity_signals.csv
```

These signals highlight product combinations frequently adopted by valuable customers.

This enables the orchestrator to identify **cross-sell and bundle opportunities**.

---

# 🔎 Data Validation and Operational Safety

The loader also generates **system diagnostics**.

These include:

```
loader_counts
validation_warnings
data_snapshot_loaded_at
```

### Loader Counts

Track how many records were loaded for each dataset.

Example:

```
customers: 200
transactions: 1,500
feature_usage: 3,200
```

This helps operators immediately confirm that data volumes look reasonable.

---

### Validation Warnings

If critical files are missing or empty, warnings are generated.

For example:

```
baseline/customers.csv missing or empty
```

This ensures the orchestrator never silently runs on incomplete data.

---

### Snapshot Timestamp

```
data_snapshot_loaded_at
```

Records the exact moment the dataset snapshot was loaded.

This allows future runs to be **compared and reproduced**.

---

# 🏢 Why This Design Reassures Executives

Many AI systems jump directly into modeling and insights.

They rarely expose how the data entering the system is validated.

That lack of transparency creates risk.

This loader addresses that concern directly.

It ensures the system can answer fundamental operational questions such as:

• What data did the system use?
• How much data was loaded?
• Were any datasets missing?
• When was the snapshot taken?

These answers create **confidence that the intelligence produced by the system is grounded in verified inputs**.

---

# ⚠ Potential Enhancements

The design is already strong, but a few improvements could further strengthen it.

### Schema Validation

Introduce optional schema checks to confirm expected fields exist.

Example:

```
customers.csv must contain Customer_ID
```

This would prevent silent errors caused by malformed files.

---

### Data Version Tracking

Store dataset version hashes or commit IDs.

This would allow exact reproduction of historical analysis runs.

---

### Numeric Type Conversion

Currently CSV values are loaded as strings.

Converting fields such as revenue, margin, or usage counts to numeric types during ingestion would simplify downstream analytics.

---

# ⭐ Final Assessment

This module demonstrates a mature understanding of how **real-world analytics systems operate**.

Its key strengths include:

• centralized data ingestion
• flexible path resolution
• structured dataset architecture
• built-in validation signals
• reproducible snapshot tracking

These characteristics ensure the orchestrator is not just an AI experiment.

It is a **reliable analytical system capable of supporting strategic decision-making**.


In [ ]:
"""
Load all PCFD v2 data from agents/data: baseline, enrichment, trends, signals.
Returns normalized dict with lists + loader_counts + data_snapshot_loaded_at + validation_warnings.
"""

import csv
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional


def _resolve_data_root(config_data_dir: str, project_root: Optional[str]) -> Path:
    """Resolve absolute path to data directory (agents/data or project_root/agents/data)."""
    if project_root:
        base = Path(project_root)
    else:
        # From agents/PCFDv2/orchestrator/utilities/data_loading.py -> 5 levels up
        base = Path(__file__).resolve().parent.parent.parent.parent.parent
    path = base / config_data_dir
    return path.resolve()


def _load_csv(path: Path) -> List[Dict[str, Any]]:
    """Load CSV with headers into list of dicts. Returns [] if file missing."""
    if not path.exists():
        return []
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(dict(row))
    return rows


def _load_json(path: Path) -> List[Dict[str, Any]]:
    """Load JSON (list or dict) into list of dicts. Returns [] if file missing."""
    if not path.exists():
        return []
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        return [data]
    return []


def load_all_pcfd_data(
    data_dir: str,
    baseline_dir: str,
    enrichment_dir: str,
    trends_dir: str,
    signals_dir: str,
    project_root: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Load all PCFD v2 datasets. Paths are relative to project_root/data_dir.

    Returns:
        out: dict with
            - customers, product_catalog, transactions
            - customer_metrics, product_metrics, feature_usage, customer_journeys, market_signals
            - product_adoption_history, segment_growth_history, feature_demand_history
            - bundle_opportunity_signals
            - loader_counts: { "customers": N, ... }
            - data_snapshot_loaded_at: UTC ISO timestamp
            - validation_warnings: list of str
    """
    root = _resolve_data_root(data_dir, project_root)
    warnings: List[str] = []
    counts: Dict[str, int] = {}
    snapshot_at = datetime.now(timezone.utc).isoformat()

    baseline = root / baseline_dir
    enrichment = root / enrichment_dir
    trends = root / trends_dir
    signals_root = root / signals_dir

    # Baseline
    customers = _load_csv(baseline / "customers.csv")
    product_catalog = _load_csv(baseline / "product_catalog.csv")
    transactions = _load_csv(baseline / "transactions.csv")
    counts["customers"] = len(customers)
    counts["product_catalog"] = len(product_catalog)
    counts["transactions"] = len(transactions)
    if not customers and baseline.exists():
        warnings.append("baseline/customers.csv missing or empty")
    if not product_catalog and baseline.exists():
        warnings.append("baseline/product_catalog.csv missing or empty")

    # Enrichment
    customer_metrics = _load_csv(enrichment / "customer_metrics.csv")
    product_metrics = _load_csv(enrichment / "product_metrics.csv")
    feature_usage = _load_csv(enrichment / "feature_usage.csv")
    customer_journeys_raw = _load_json(enrichment / "customer_journeys.json")
    market_signals_raw = _load_json(enrichment / "market_signals.json")
    customer_journeys = customer_journeys_raw if isinstance(customer_journeys_raw, list) else [customer_journeys_raw] if customer_journeys_raw else []
    market_signals = market_signals_raw if isinstance(market_signals_raw, list) else [market_signals_raw] if market_signals_raw else []
    counts["customer_metrics"] = len(customer_metrics)
    counts["product_metrics"] = len(product_metrics)
    counts["feature_usage"] = len(feature_usage)
    counts["customer_journeys"] = len(customer_journeys)
    counts["market_signals"] = len(market_signals)

    # Trends
    product_adoption_history = _load_csv(trends / "product_adoption_history.csv")
    segment_growth_history = _load_csv(trends / "segment_growth_history.csv")
    feature_demand_history = _load_csv(trends / "feature_demand_history.csv")
    counts["product_adoption_history"] = len(product_adoption_history)
    counts["segment_growth_history"] = len(segment_growth_history)
    counts["feature_demand_history"] = len(feature_demand_history)

    # Signals
    bundle_opportunity_signals = _load_csv(signals_root / "bundle_opportunity_signals.csv")
    counts["bundle_opportunity_signals"] = len(bundle_opportunity_signals)

    return {
        "customers": customers,
        "product_catalog": product_catalog,
        "transactions": transactions,
        "customer_metrics": customer_metrics,
        "product_metrics": product_metrics,
        "feature_usage": feature_usage,
        "customer_journeys": customer_journeys,
        "market_signals": market_signals,
        "product_adoption_history": product_adoption_history,
        "segment_growth_history": segment_growth_history,
        "feature_demand_history": feature_demand_history,
        "bundle_opportunity_signals": bundle_opportunity_signals,
        "loader_counts": counts,
        "data_snapshot_loaded_at": snapshot_at,
        "validation_warnings": warnings,
    }
